# Tutorial 01: Clasificación de Arritmias ECG y Post-Training Quantization (PTQ)
### TinyML para Raspberry Pi Pico 2 / Pico 2 W (RP2350 - Cortex-M33) con TensorFlow y Arduino

En este tutorial aprenderás:
1. **Generación de bioseñales sintéticas** usando `neurokit2` (Ritmo Sinusal Normal, Taquicardia y Bradicardia/Ectópico).
2. **Diseño de una arquitectura 1D-CNN ultra-compacta** basada en convoluciones separables en profundidad (Depthwise-Separable Convolutions).
3. **Fundamentos matemáticos de la Cuantización Post-Entrenamiento (PTQ)** a enteros de 8 bits (INT8):
   $$q = \text{clip}\left(\text{round}\left(\frac{x}{S}\right) + Z, -128, 127\right)$$
4. **Calibración con Representative Dataset** para mapear los rangos dinámicos reales de activación.
5. **Exportación directa a un arreglo en C (`model_data.h`)** y muestras de prueba (`test_samples.h`) para inferencia en tiempo real en la Raspberry Pi Pico 2.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Añadir directorio raíz del proyecto
PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.signal_generator import generate_ecg_dataset
from utils.exporter import tflite_to_c_header, save_test_samples_header
from utils.tflite_eval import evaluate_tflite_model

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

TensorFlow Version: 2.16.2
GPU disponible: []


## 1. Generación de Señales Sintéticas de ECG con NeuroKit2
Generamos ventanas de 187 muestras a 250 Hz (estilo MIT-BIH Arrhythmia Database). Cada latido incluye variabilidad cardíaca y ruido fisiológico realista.

In [2]:
SAMPLE_LEN = 187
SAMPLING_RATE = 250
CLASS_NAMES = ["Normal", "Tachycardia", "Bradycardia"]

In [3]:

# X, y = generate_ecg_dataset(num_samples_per_class=400, sample_len=SAMPLE_LEN, sampling_rate=SAMPLING_RATE)
# X = np.expand_dims(X, axis=-1) # Forma (N, 187, 1)

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
# print(f"Dimensiones de X_train: {X_train.shape} | X_test: {X_test.shape}")

# # Visualización de latidos sintéticos representativos
# fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
# for c_idx, c_name in enumerate(CLASS_NAMES):
#     idx = np.where(y_train == c_idx)[0][0]
#     axes[c_idx].plot(X_train[idx, :, 0], color='tab:blue' if c_idx==0 else ('tab:red' if c_idx==1 else 'tab:orange'))
#     axes[c_idx].set_title(f"Clase {c_idx}: {c_name}", fontsize=12, fontweight='bold')
#     axes[c_idx].set_xlabel("Muestra")
#     axes[c_idx].grid(True, linestyle="--", alpha=0.6)
# axes[0].set_ylabel("Amplitud Normalizada")
# plt.tight_layout()
# plt.show()

### optional saving the data

In [4]:
# Save all arrays into a single compressed file
#np.savez_compressed(
#    "ecg_dataset.npz",
#    X_train=X_train,
#    X_test=X_test,
#    y_train=y_train,
#    y_test=y_test,
#)
#print("Dataset successfully saved as ecg_dataset.npz")

In [5]:
# Load the archive
data = np.load("ecg_dataset.npz")

# Access individual arrays
X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

print("Dataset successfully loaded!")
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

Dataset successfully loaded!
X_train shape: (900, 187, 1) | X_test shape: (300, 187, 1)


## 2. Construcción del Modelo 1D-CNN (Optimizado para Microcontrolador RP2350)
Para operar en microcontroladores con presupuestos estrictos de memoria SRAM y Flash, utilizamos **SeparableConv1D** (convolución separable espacial/canal), lo cual reduce los parámetros en más del 70% comparado con convoluciones 1D tradicionales.

In [6]:
def build_tiny_ecg_model(input_shape=(SAMPLE_LEN, 1), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape, name="ecg_input")
    
    # Bloque 1
    x = tf.keras.layers.Conv1D(filters=8, kernel_size=7, strides=2, padding="same", use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    
    # Bloque 2: Depthwise Separable
    x = tf.keras.layers.SeparableConv1D(filters=16, kernel_size=5, strides=2, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    
    # Bloque 3: Depthwise Separable
    x = tf.keras.layers.SeparableConv1D(filters=32, kernel_size=3, strides=2, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    
    # Global Pooling + Clasificador
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="probabilities")(x)
    
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="tiny_ecg_cnn")

model = build_tiny_ecg_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "tiny_ecg_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ecg_input (InputLayer)          │ (None, 187, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 94, 8)          │            56 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 94, 8)          │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 94, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv1d                │ (None, 47, 16)         │           168 │
│ (SeparableConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 47, 16)         │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 47, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv1d_1              │ (None, 24, 32)         │           560 │
│ (SeparableConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ probabilities (Dense)           │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,107 (4.32 KB)

 Trainable params: 995 (3.89 KB)

 Non-trainable params: 112 (448.00 B)

## 3. Entrenamiento del Modelo Base

In [7]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nExactitud en Test (Float32): {test_acc * 100:.2f}%")

Epoch 1/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5611 - loss: 0.9447 - val_accuracy: 0.3333 - val_loss: 1.0939
Epoch 2/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7456 - loss: 0.6993 - val_accuracy: 0.3333 - val_loss: 1.0868
Epoch 3/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7711 - loss: 0.5610 - val_accuracy: 0.3333 - val_loss: 1.1027
Epoch 4/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8156 - loss: 0.4805 - val_accuracy: 0.3333 - val_loss: 1.1583
Epoch 5/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8500 - loss: 0.4079 - val_accuracy: 0.3333 - val_loss: 1.2892
Epoch 6/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8556 - loss: 0.3829 - val_accuracy: 0.3333 - val_loss: 1.4233
Epoch 7/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8689 - loss: 0.3496 - val_accuracy: 0.3333 - val_loss: 1.4228
Epoch 8/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8967 - loss: 0.3015 - val_accuracy: 0.3333 - va

## 4. Conversión a LiteRT para microcontroladores (PTQ Full INT8)
TensorFlow Lite ahora se presenta como **LiteRT**; para microcontroladores, el runtime es **LiteRT for Microcontrollers** (antes TensorFlow Lite Micro / TFLM). El flujo compatible con el Pico sigue siendo: modelo TensorFlow → archivo `.tflite` (FlatBuffer) → arreglo C/C++ (`model_data.h`) → firmware que enlaza el runtime de microcontroladores.

Usaremos un **SavedModel como entrada del conversor**, una ruta recomendada por la documentación actual de LiteRT que separa la exportación Keras de la conversión. La cuantización Full INT8 requiere un conjunto representativo; el modelo resultante debe usar operaciones disponibles en LiteRT for Microcontrollers y en la biblioteca/runtime configurada para Arduino.

In [8]:
import tempfile

def representative_data_gen():
    for sample in X_train[:200]:
        yield [sample[np.newaxis, ...].astype(np.float32)]

# Keras 3 exports SavedModel; LiteRT converts that stable on-disk format.
with tempfile.TemporaryDirectory(prefix="ecg_litert_") as temp_dir:
    saved_model_dir = Path(temp_dir) / "saved_model"
    model.export(str(saved_model_dir))

    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    tflite_int8 = converter.convert()

    # ArduTFLite exposes a float input/output API, so create a matching FP32 model.
    converter_float = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    tflite_float = converter_float.convert()

ARDUINO_DIR = Path("../arduino/ecg_inference_pico2")
ARDUINO_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARDUINO_DIR / "model_ecg_int8.tflite"
model_path.write_bytes(tflite_int8)
print(f"LiteRT INT8 model saved: {model_path} ({len(tflite_int8) / 1024:.2f} KB)")

ADC_ARDUINO_DIR = Path("../arduino/ecg_adc_inference_pico")
ADC_ARDUINO_DIR.mkdir(parents=True, exist_ok=True)
float_model_path = ADC_ARDUINO_DIR / "model_ecg_float.tflite"
float_model_path.write_bytes(tflite_float)
tflite_to_c_header(
    tflite_float,
    str(ADC_ARDUINO_DIR / "model_data.h"),
    array_name="g_model",
)
print(f"ArduTFLite FP32 model saved: {float_model_path} ({len(tflite_float) / 1024:.2f} KB)")

INFO:tensorflow:Assets written to: C:\Users\47470482\AppData\Local\Temp\ecg_litert_vvbwin47\saved_model\assets


INFO:tensorflow:Assets written to: C:\Users\47470482\AppData\Local\Temp\ecg_litert_vvbwin47\saved_model\assets


Saved artifact at 'C:\Users\47470482\AppData\Local\Temp\ecg_litert_vvbwin47\saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 187, 1), dtype=tf.float32, name='ecg_input')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2354118110928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118114000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118113232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118113808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118113040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118113424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118114384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118114192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118115152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2354118113616: TensorSpec(shape=(), dtype=tf.resource, nam

## 5. Evaluación y empaquetado para Arduino (`model_data.h`)
El archivo `.h` solo contiene los bytes del modelo. El sketch de ADC usa la API `float` de **ArduTFLite** (`modelInit`, `modelSetInput`, `modelRunInference`, `modelGetOutput`), por lo que este firmware carga el modelo Float32 generado en la carpeta `ecg_adc_inference_pico`. El modelo INT8 continúa disponible para el sketch de validación anterior. Instala ArduTFLite y su dependencia `Chirale_TensorFlowLite`; comprueba que la versión/fork instalada sea compatible con tu placa RP2040.

In [9]:
res_int8 = evaluate_tflite_model(tflite_int8, X_test, y_test)
print(f"Exactitud Modelo Cuantizado INT8: {res_int8['accuracy'] * 100:.2f}%")
print(f"Escala de Entrada: {res_int8['input_scale']:.6f} | Zero-Point: {res_int8['input_zero_point']}")
if not res_int8["is_int8"]:
    raise TypeError("El modelo exportado no tiene entrada INT8; revisa la conversión Full INT8.")

res_float = evaluate_tflite_model(tflite_float, X_test, y_test)
print(f"Exactitud Modelo Float32 (ArduTFLite): {res_float['accuracy'] * 100:.2f}%")
if res_float["is_int8"]:
    raise TypeError("ArduTFLite necesita el modelo Float32 exportado en este notebook.")

# Guardar en el directorio de Arduino del Ejemplo 01
ARDUINO_DIR = Path("../arduino/ecg_inference_pico2")
ARDUINO_DIR.mkdir(parents=True, exist_ok=True)

tflite_to_c_header(tflite_int8, str(ARDUINO_DIR / "model_data.h"), array_name="g_model")
save_test_samples_header(X_test[:, :, 0], y_test, str(ARDUINO_DIR / "test_samples.h"), CLASS_NAMES, samples_per_class=3)
print("Archivos C listos para compilar en Arduino IDE!")

Exactitud Modelo Cuantizado INT8: 78.67%
Escala de Entrada: 0.030570 | Zero-Point: -56
[Exporter] Successfully exported C Header: ..\arduino\ecg_inference_pico2\model_data.h (10080 bytes)
[Exporter] Saved 9 test samples to ..\arduino\ecg_inference_pico2\test_samples.h
Archivos C listos para compilar en Arduino IDE!
